# Week 2 — Dataset Acquisition and Cleaning
**Project:** Small but Sufficient? — SLM vs. LLM phishing-email detection

This notebook builds your real, locked train/validation/test dataset from a verified, already-cleaned public source, replacing the synthetic placeholder used in the Week 5/6 fine-tuning notebook.

**What changed from the original plan:** instead of separately downloading Nazario, Enron, and CEAS 2008 as raw mbox/CSV files from various personal sites and Kaggle, this uses a single verified Hugging Face dataset — `puyang2025/seven-phishing-email-datasets` — that already combines Enron, CEAS-08, SpamAssassin ("Assassin"), TREC-05/06/07, and Ling-Spam into one clean, unified schema (203,017 rows total). This is lower-risk (no dead links, no manual mbox parsing) and gives you *more* source diversity than the original 3-corpus plan, not less.

**Honest limitation to note in your paper's Data section:** this collection does not include the Nazario corpus specifically. TREC-05/06/07 and SpamAssassin are equally standard, widely-used phishing/spam benchmarks in the literature, so this is a defensible substitution, not a downgrade — but say so explicitly rather than implying you used Nazario when you didn't.

**Steps:**
1. Runtime → Change runtime type → CPU is fine for this notebook (no GPU needed for data work).
2. Run top to bottom.
3. End state: three locked files in Drive — `phishing_train.csv`, `phishing_val.csv`, `phishing_test.csv`. **Do not open or inspect `phishing_test.csv` again until Week 8.**


In [ ]:
# CELL 1 — Install dependencies
!pip install -q datasets pandas scikit-learn


In [ ]:
# CELL 2 — Mount Google Drive
import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
else:
    DRIVE_ROOT = "."

os.makedirs(f"{DRIVE_ROOT}/data/processed", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/data/raw", exist_ok=True)
print("Project root:", DRIVE_ROOT)


In [ ]:
# CELL 3 — Imports
import pandas as pd
import numpy as np
import hashlib
from datasets import load_dataset
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)


## Config

`SOURCES_TO_USE` controls which of the 7 sub-corpora go into your dataset.

- **`["Enron", "CEAS-08"]`** — matches your original 2-corpus plan most closely (legitimate + phishing/spam).
- **`["Enron", "CEAS-08", "Assassin", "TREC-05", "TREC-06", "TREC-07"]`** — recommended: more data, more source diversity, still excludes `Ling` (smallest, least commonly cited in recent phishing-LLM papers).
- All 7 — maximum data (203k rows) but includes the smaller, less-standard Ling-Spam corpus too.

Start with the recommended 6-source set below; you can always narrow it later if training time becomes a bottleneck.


In [ ]:
# CELL 4 — Config
SOURCES_TO_USE = ["Enron", "CEAS-08", "Assassin", "TREC-05", "TREC-06", "TREC-07"]

MAX_ROWS_PER_SOURCE = 5000   # cap per source so no single corpus dominates; set to None for no cap
RANDOM_SEED = 42

TRAIN_FRAC = 0.70
VAL_FRAC = 0.10
TEST_FRAC = 0.20

OUT_TRAIN = f"{DRIVE_ROOT}/data/processed/phishing_train.csv"
OUT_VAL   = f"{DRIVE_ROOT}/data/processed/phishing_val.csv"
OUT_TEST  = f"{DRIVE_ROOT}/data/processed/phishing_test.csv"
SUMMARY_PATH = f"{DRIVE_ROOT}/data/data_summary.md"

print("Using sources:", SOURCES_TO_USE)


In [ ]:
# CELL 5 — Load the dataset from Hugging Face
ds = load_dataset("puyang2025/seven-phishing-email-datasets")
df_all = ds["train"].to_pandas()
print("Total rows loaded:", len(df_all))
df_all["dataset_name"].value_counts()


In [ ]:
# CELL 6 — Filter to chosen sources
df = df_all[df_all["dataset_name"].isin(SOURCES_TO_USE)].copy()
print("Rows after source filter:", len(df))
df["dataset_name"].value_counts()


## Cleaning

Steps, in order:
1. Drop rows with missing body text or missing label (can't use these).
2. Cap rows per source (if configured) so one corpus doesn't dominate the dataset.
3. Deduplicate exact-duplicate email bodies.
4. Flag (not drop) near-duplicate quoted-thread text — the dataset card explicitly warns about this as a train/test leakage risk, so we handle it before splitting, not after.


In [ ]:
# CELL 7 — Clean: drop missing, cap per source
before = len(df)
df = df.dropna(subset=["text", "label"])
df = df[df["text"].str.strip().str.len() > 0]
print(f"Dropped {before - len(df)} rows with missing/empty body or label.")

if MAX_ROWS_PER_SOURCE is not None:
    df = (
        df.groupby("dataset_name", group_keys=False)
          .apply(lambda g: g.sample(n=min(len(g), MAX_ROWS_PER_SOURCE), random_state=RANDOM_SEED))
          .reset_index(drop=True)
    )
    print("Rows after per-source cap:", len(df))

df["dataset_name"].value_counts()


In [ ]:
# CELL 8 — Deduplicate
before = len(df)
df["body_hash"] = df["text"].apply(lambda t: hashlib.md5(t.strip().lower().encode()).hexdigest())
df = df.drop_duplicates(subset="body_hash").drop(columns="body_hash")
print(f"Dropped {before - len(df)} exact-duplicate emails.")
print("Rows remaining:", len(df))


In [ ]:
# CELL 9 — Rename to match the fine-tuning notebook's expected schema
# The Week 5/6 notebook expects columns: subject, body, label
df = df.rename(columns={"text": "body"})
df["subject"] = df["subject"].fillna("")
df["email_id"] = range(len(df))

final_cols = ["email_id", "subject", "body", "label", "dataset_name"]
df = df[final_cols]

print(df["label"].value_counts(normalize=True).rename({0: "legitimate/ham", 1: "phishing/spam"}))
df.head()


## Split

Stratified by `label` **and** `dataset_name`, so each split has a representative mix of both classes and all sources — this matters because class balance and source composition differ a lot per corpus (see the per-source counts above), and an unstratified split could accidentally put almost all of one source into the test set.

**After this cell runs, `phishing_test.csv` is locked. Do not load, inspect, or reference it again until Week 8's main experiment — not even "just to check the shape."**


In [ ]:
# CELL 10 — Stratified train/val/test split
# stratify on a combined key so both label and source distribution are preserved across splits
df["strat_key"] = df["label"].astype(str) + "_" + df["dataset_name"]

train_df, temp_df = train_test_split(
    df, test_size=(VAL_FRAC + TEST_FRAC), stratify=df["strat_key"], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=(TEST_FRAC / (VAL_FRAC + TEST_FRAC)), stratify=temp_df["strat_key"], random_state=RANDOM_SEED
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(split)} rows, {split['label'].mean():.1%} phishing/spam")

train_df = train_df.drop(columns="strat_key")
val_df = val_df.drop(columns="strat_key")
test_df = test_df.drop(columns="strat_key")


In [ ]:
# CELL 11 — Save to Drive
train_df.to_csv(OUT_TRAIN, index=False)
val_df.to_csv(OUT_VAL, index=False)
test_df.to_csv(OUT_TEST, index=False)

print("Saved:")
print(" ", OUT_TRAIN, f"({len(train_df)} rows)")
print(" ", OUT_VAL, f"({len(val_df)} rows)")
print(" ", OUT_TEST, f"({len(test_df)} rows)  <-- LOCKED, do not touch until Week 8")


In [ ]:
# CELL 12 — Write a data summary (becomes part of your Week 12 Data section)
summary_lines = [
    "# Data Summary — Week 2\n",
    f"Source: `puyang2025/seven-phishing-email-datasets` (Hugging Face)\n",
    f"Sub-corpora used: {', '.join(SOURCES_TO_USE)}\n",
    f"Per-source cap: {MAX_ROWS_PER_SOURCE if MAX_ROWS_PER_SOURCE else 'none'}\n",
    f"Random seed: {RANDOM_SEED}\n\n",
    "## Split sizes\n",
    f"- Train: {len(train_df)} rows\n",
    f"- Validation: {len(val_df)} rows\n",
    f"- Test (locked): {len(test_df)} rows\n\n",
    "## Class balance (phishing/spam = 1)\n",
    f"- Train: {train_df['label'].mean():.1%}\n",
    f"- Validation: {val_df['label'].mean():.1%}\n",
    f"- Test: {test_df['label'].mean():.1%}\n\n",
    "## Known limitations (state these explicitly in the paper)\n",
    "- Does not include the Nazario corpus specifically; TREC-05/06/07 and SpamAssassin substitute as comparably standard benchmarks.\n",
    "- Source dataset's label semantics conflate general spam with phishing specifically for some sources — not a pure phishing-only signal.\n",
    "- Exact-duplicate emails removed; near-duplicate quoted-thread text (e.g. reply chains) was NOT specifically deduplicated beyond exact-match — a residual leakage risk between splits, worth flagging as a limitation.\n",
    "- Emails are historical (sources span roughly 1999-2008); modern phishing style may differ, which is exactly why the Week 7 adversarial (LLM-generated) test set matters.\n",
]

with open(SUMMARY_PATH, "w") as f:
    f.writelines(summary_lines)

print("Saved summary to:", SUMMARY_PATH)
print("".join(summary_lines))


---
### Next steps
- Point the Week 5/6 fine-tuning notebook's `DATA_PATH` at `phishing_train.csv` (never `phishing_test.csv`).
- Use `phishing_val.csv` for any tuning/early checks during Weeks 3-7.
- `phishing_test.csv` stays locked until Week 8.
- Copy `data_summary.md`'s content into your repo's `data/` folder and commit it alongside these three CSVs (the CSVs themselves live in Drive per your `.gitignore`, but the summary is small text and safe to commit to GitHub).
